# Metaclasses
In Python, everything is an object, including classes themselves. A metaclass is a "class of a class." It defines how a class behaves and how it is created. The default metaclass in Python is `type`.

## `type()` as a Class Factory
You can use `type()` to create a class dynamically at runtime.

In [2]:
# type(name, bases, dict)
MyDynamicClass = type("MyDynamicClass", (), {"x": 5, "greet": lambda self: "Hello!"})

obj = MyDynamicClass()
print(obj.x)
print(obj.greet())

5
Hello!


## Custom Metaclass
To create a custom metaclass, inherit from `type` and override `__new__` (which creates the class object) or `__init__` (which initializes the class object).

In [3]:
class Singleton(type):
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class Database(metaclass=Singleton):
    def __init__(self):
        print("Connecting to DB...")

db1 = Database()
db2 = Database()
print(f"Same instance? {db1 is db2}")

Connecting to DB...
Same instance? True


## Practical Example: Attribute Validation
Metaclasses can ensure that subclasses follow specific rules at the time they are defined.

In [4]:
class UpperCaseAttributes(type):
    def __new__(cls, name, bases, dct):
        # Transform any non-magic attributes to uppercase
        uppercase_dct = {}
        for key, value in dct.items():
            if not key.startswith("__"):
                uppercase_dct[key.upper()] = value
            else:
                uppercase_dct[key] = value
        return super().__new__(cls, name, bases, uppercase_dct)

class MyApp(metaclass=UpperCaseAttributes):
    version = "1.0"
    def run(self):
        pass

print(hasattr(MyApp, "version")) # False
print(hasattr(MyApp, "VERSION")) # True

False
True


## `__init_subclass__` (The Modern Alternative)
In many cases, you don't actually need a metaclass. `__init_subclass__` (Python 3.6+) is a simpler way to customize class creation.

In [5]:
class PluginBase:
    plugins = []
    
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.plugins.append(cls)

class AudioPlugin(PluginBase): pass
class VideoPlugin(PluginBase): pass

print(PluginBase.plugins)

[<class '__main__.AudioPlugin'>, <class '__main__.VideoPlugin'>]


## Summary
- **Metaclass**: A class used to create other classes.
- **Metaclass Inheritance**: Inherit from `type`.
- **Selection**: Use `metaclass=MyMeta` in the class definition.
- **Philosophy**: "Metaclasses are deeper magic than 99% of users should ever worry about. If you wonder whether you need them, you don't." — Tim Peters.